In [2]:
# ========================================
# 📌 1. IMPORTS & SETUP
# ========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
from gensim import corpora, models
from collections import Counter
from datetime import datetime

from sklearn.feature_extraction.text import CountVectorizer

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# %matplotlib inline
sns.set(style="whitegrid")

# ========================================
# 📌 2. LOAD DATA
# ========================================
file_path = r'C:\Users\ayedr\week-1\notebooks\raw_analyst_ratings.xlsx'
df = pd.read_excel(file_path)
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df['date'] = pd.to_datetime(df['date'], utc=True)
news_df = df.copy()

# Clean and tokenize
news_df['tokens'] = news_df['headline'].astype(str).str.lower().str.split()
news_df['tokens'] = news_df['tokens'].apply(
    lambda x: [w for w in x if w not in stop_words and len(w) > 2]
)

# Create dictionary and corpus for LDA
dictionary = corpora.Dictionary(news_df['tokens'])
corpus = [dictionary.doc2bow(text) for text in news_df['tokens']]

# LDA model (adjust num_topics if needed)
lda_model = models.LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15)

# Print discovered topics
for idx, topic in lda_model.print_topics():
    print(f"🔹 Topic {idx+1}:\n{topic}\n")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ayedr\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


🔹 Topic 1:
0.074*"stocks" + 0.031*"earnings" + 0.023*"52-week" + 0.021*"session" + 0.021*"moving" + 0.021*"top" + 0.016*"scheduled" + 0.014*"pre-market" + 0.013*"watch" + 0.012*"new"

🔹 Topic 2:
0.014*"movers" + 0.014*"qualcomm" + 0.013*"says" + 0.010*"option" + 0.010*"biggest" + 0.009*"alert:" + 0.009*"stock" + 0.008*"yesterday" + 0.007*"earnings" + 0.007*"qihoo"

🔹 Topic 3:
0.072*"eps" + 0.060*"reports" + 0.039*"sales" + 0.028*"est" + 0.021*"est." + 0.020*"sees" + 0.015*"revenue" + 0.015*"adj." + 0.014*"est;" + 0.012*"vs."

🔹 Topic 4:
0.042*"shares" + 0.026*"market" + 0.021*"benzinga's" + 0.019*"trading" + 0.015*"update:" + 0.014*"higher" + 0.013*"top" + 0.012*"companies" + 0.012*"oil" + 0.010*"downgrades"

🔹 Topic 5:
0.028*"announces" + 0.020*"maintains" + 0.019*"target" + 0.019*"raises" + 0.019*"price" + 0.017*"buy" + 0.016*"upgrades" + 0.014*"downgrades" + 0.014*"initiates" + 0.013*"lowers"



In [9]:
df['date'] = pd.to_datetime(df['date']).dt.date

# Install if not already available (uncomment if needed)
# !pip install vaderSentiment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize the analyzer
analyzer = SentimentIntensityAnalyzer()

# Define a function to get sentiment scores
def get_vader_sentiment(text):
    scores = analyzer.polarity_scores(str(text))
    return scores['compound']  # compound score summarizes overall sentiment

# Apply to the headlines
news_df['sentiment'] = news_df['headline'].apply(get_vader_sentiment)
# Show a sample of the relevant columns
news_df[['headline', 'date', 'stock', 'sentiment']].head(10)


,headline,date,stock,sentiment
0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 14:30:54+00:00,A,0.000
1,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 14:45:20+00:00,A,0.000
2,71 Biggest Movers From Friday,2020-05-26 08:30:07+00:00,A,0.000
3,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 16:45:06+00:00,A,0.000
4,B of A Securities Maintains Neutral on Agilent...,2020-05-22 15:38:59+00:00,A,0.296
5,"CFRA Maintains Hold on Agilent Technologies, L...",2020-05-22 15:23:25+00:00,A,-0.128
6,"UBS Maintains Neutral on Agilent Technologies,...",2020-05-22 13:36:20+00:00,A,0.000
7,Agilent Technologies shares are trading higher...,2020-05-22 13:07:04+00:00,A,0.296
8,Wells Fargo Maintains Overweight on Agilent Te...,2020-05-22 12:37:59+00:00,A,-0.128
9,10 Biggest Price Target Changes For Friday,2020-05-22 12:06:17+00:00,A,0.000


In [10]:
# Group by stock and date to get average daily sentiment
daily_sentiment = news_df.groupby(['stock', 'date'])['sentiment'].mean().reset_index()
daily_sentiment.rename(columns={'sentiment': 'avg_sentiment'}, inplace=True)

# Preview the result
daily_sentiment.head()


,stock,date,avg_sentiment
0,A,2009-04-29 00:00:00+00:00,0.0000
1,A,2009-06-01 00:00:00+00:00,0.2960
2,A,2009-07-14 00:00:00+00:00,0.3818
3,A,2009-07-30 00:00:00+00:00,0.0000
4,A,2009-08-04 00:00:00+00:00,0.0000


In [19]:
import os

# Folder where the stock files are stored
stock_folder = r'C:\Users\ayedr\week-1\data\yfinance_data'

def load_and_compute_returns(stock_symbol):
    file_path = os.path.join(stock_folder, f"{stock_symbol}.xlsx")
    df = pd.read_excel(file_path)

    df['Date'] = pd.to_datetime(df['Date']).dt.date
    df.sort_values('Date', inplace=True)
    df['daily_return'] = df['Close'].pct_change()

    # Clean symbol for consistency
    clean_symbol = stock_symbol.replace('_historical_data', '')
    df['stock'] = clean_symbol

    return df[['Date', 'stock', 'daily_return']].dropna()


# Example: Load AAPL and preview
aapl_returns = load_and_compute_returns('AAPL_historical_data')
print(aapl_returns.head())

stocks = ['AAPL_historical_data', 'TSLA_historical_data', 'GOOG_historical_data', 'MSFT_historical_data', 'AMZN_historical_data', 'NVDA_historical_data']
returns_list = [load_and_compute_returns(symbol) for symbol in stocks]

# Combine all into a single DataFrame
all_returns_df = pd.concat(returns_list, ignore_index=True)
print(all_returns_df.head())


         Date stock  daily_return
1  1980-12-15  AAPL     -0.052171
2  1980-12-16  AAPL     -0.073398
3  1980-12-17  AAPL      0.024751
4  1980-12-18  AAPL      0.028992
5  1980-12-19  AAPL      0.061029
         Date stock  daily_return
0  1980-12-15  AAPL     -0.052171
1  1980-12-16  AAPL     -0.073398
2  1980-12-17  AAPL      0.024751
3  1980-12-18  AAPL      0.028992
4  1980-12-19  AAPL      0.061029


In [21]:
stocks = ['AAPL_historical_data', 'TSLA_historical_data', 'GOOG_historical_data',
          'MSFT_historical_data', 'AMZN_historical_data', 'NVDA_historical_data']
returns_list = [load_and_compute_returns(s) for s in stocks]

all_returns_df = pd.concat(returns_list, ignore_index=True)
print(all_returns_df['stock'].unique())




['AAPL' 'TSLA' 'GOOG' 'MSFT' 'AMZN' 'NVDA']


In [22]:
# Filter sentiment data to just these 6 stocks
target_stocks = ['AAPL', 'TSLA', 'GOOG', 'MSFT', 'AMZN', 'NVDA']
avg_sentiment_df = avg_sentiment_df[avg_sentiment_df['stock'].isin(target_stocks)]

# Normalize date formats
avg_sentiment_df['date'] = pd.to_datetime(avg_sentiment_df['date']).dt.date
all_returns_df['date'] = pd.to_datetime(all_returns_df['Date']).dt.date

# Merge on stock and date
merged_df = pd.merge(avg_sentiment_df, all_returns_df, on=['stock', 'date'])

# Preview result
print(merged_df.head())


  stock        date  avg_sentiment        Date  daily_return
0  AAPL  2020-03-09      -0.302067  2020-03-09     -0.079092
1  AAPL  2020-03-10      -0.090787  2020-03-10      0.072022
2  AAPL  2020-03-11      -0.023850  2020-03-11     -0.034731
3  AAPL  2020-03-12      -0.207240  2020-03-12     -0.098755
4  AAPL  2020-03-13      -0.023191  2020-03-13      0.119808


In [18]:
# Compute correlation per stock
correlation_results = (
    merged_df.groupby('stock')[['avg_sentiment', 'daily_return']]
    .corr()
    .iloc[0::2, -1]  # Get correlation between avg_sentiment and daily_return
    .reset_index()
    .drop(columns='level_1')
    .rename(columns={'daily_return': 'pearson_correlation'})
)

# Display result
print(correlation_results)


Sentiment stock names: ['A' 'AA' 'AAC' ... 'QKOR' 'QLD' 'QLGC']
Returns stock names: ['AAPL_historical_data' 'TSLA_historical_data' 'GOOG_historical_data'
 'MSFT_historical_data' 'AMZN_historical_data' 'NVDA_historical_data']
